# Lab 02：Baseline anomaly detection 與 alerting

Lab 01 留下一組 feature。這一節把 feature 變成 score、score 變成決策、決策變成 alert，最後量這一整套值不值得交給值班人員。

三層要分清楚，因為損害發生在不同層，修的地方也不同：

| 層 | 是什麼 | 對「要不要通知人」的立場 |
|---|---|---|
| **score** | 連續數值，與正常狀態的距離 | 沒有立場 |
| **label** | score 跨過 threshold 並且持續了一段時間 | 統計上確認了，還沒決定通知誰 |
| **alert** | 通過維運過濾條件後真的送到人手上的那一部分 | 這一層才決定 |

| 步驟 | 做什麼 |
|---|---|
| 1 | 載入 feature 與 ground truth |
| 2 | 四種 baseline，同一個 event 四種讀法 |
| 3 | 四種 baseline 的命中與代價 |
| 4 | robust 統計的 breakdown point |
| 5 | threshold 4.0 不是單純的 tail probability |
| 6 | CUSUM：給偵測器記憶 |
| 7 | EWMA：另一種記憶 |
| 8 | change-point detection：水位移動了嗎 |
| 9 | degenerate 的欄位：統計不是工具的時候 |
| 10 | 多 feature 合成 score_max |
| 11 | score 到決策：連續 N 筆 confirmation |
| 12 | 決策到通知：alert policy |
| 13 | scorecard：event recall，不是 point accuracy |
| 14 | precision-recall 與 ROC |
| 15 | threshold × N 的二維 sweep |
| 16 | cost function：threshold 是商業決策 |
| 17 | 時間 holdout 與 confidence interval |
| 18 | weak label 下的評估偏差 |

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Georgia", "Times New Roman", "DejaVu Serif"],
    "figure.dpi": 110, "axes.titlesize": 10, "axes.titlelocation": "left",
    "axes.labelsize": 9, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
})
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "environments").is_dir():
    ROOT = ROOT.parent
DATA = ROOT / "data" / "synthetic"
OUT = ROOT / "outputs" / "workshop"

C = {"signal": "#3B7DD8", "baseline": "#E0752D", "score": "#7A5AC7",
     "alert": "#D6455D", "truth": "#FF9F1C", "peer": "#2E9E7B", "muted": "#9AA4B0"}
MAD_TO_SIGMA = 1.4826
MAX_SCORE = 50.0

# The two knobs this whole notebook is about. Step 15 sweeps them jointly; every number
# before that is measured at these values, so changing them here changes the whole lab.
THRESHOLD = 4.0
N_CONSEC = 2

## 第 1 步：載入 feature 與 ground truth

讀回 Lab 01 存的 `features.csv`，並且從原始資料重建 event window 與 change calendar。ground truth 不從 feature 檔帶過來，因為 recall 的分母應該由 ground truth 定義，不應該跟著 feature 檔一起漂。

In [ ]:
feat_path = OUT / "features.csv"
if not feat_path.exists():
    raise FileNotFoundError(f"{feat_path} not found. Run Lab 01 to completion first.")

tel = pd.read_csv(feat_path, parse_dates=["timestamp"])
tel = tel.sort_values(["port_id", "timestamp"]).reset_index(drop=True)

CADENCE = float(tel.groupby("port_id")["timestamp"].diff().dt.total_seconds().median())
WINDOW = int(round(3600 / CADENCE))
SPAN = (tel["timestamp"].min(), tel["timestamp"].max())
DAYS = (SPAN[1] - SPAN[0]).total_seconds() / 86400

# Rebuild the event windows from the label column rather than importing them from the
# feature file, so the denominator of every recall number stays tied to ground truth.
events = (tel[tel["event_label"] != "normal"]
          .groupby(["event_id", "event_label", "device_id", "port_id", "port_role"], observed=True)
          .agg(start=("timestamp", "min"), end=("timestamp", "max"), n=("timestamp", "size"))
          .reset_index().rename(columns={"event_label": "event_type"})
          .sort_values("start").reset_index(drop=True))

calendar = (pd.read_csv(DATA / "change_calendar.csv", parse_dates=["start_time", "end_time"])
              .rename(columns={"start_time": "start", "end_time": "end"}))


def in_scope(scope, device_id, port_id):
    """A calendar entry covers this port if its scope names the port, the device, or everything."""
    return str(scope) in ("ALL", "*", str(device_id), str(port_id))


planned = pd.Series(False, index=events.index)
for ch in calendar.itertuples():
    for i, e in events.iterrows():
        if in_scope(ch.scope, e.device_id, e.port_id) and e.start <= ch.end and e.end >= ch.start:
            planned.loc[i] = True
events["planned"] = planned
incidents = events[~events["planned"]].reset_index(drop=True)

FEATURES = [c[2:] for c in tel.columns if c.startswith("z_")]
ports = sorted(tel["port_id"].unique())


def shade(ax, time, mask, color=None, label="event"):
    """Shade every contiguous run of True in `mask`, collapsing them into one legend entry."""
    v = np.asarray(pd.Series(mask).fillna(False), dtype=bool)
    idx = np.flatnonzero(v)
    if idx.size == 0:
        return
    brk = np.flatnonzero(np.diff(idx) > 1)
    starts, ends = np.r_[idx[0], idx[brk + 1]], np.r_[idx[brk], idx[-1]]
    t = pd.Series(time).reset_index(drop=True)
    for n, (a, b) in enumerate(zip(starts, ends)):
        ax.axvspan(t.iloc[a], t.iloc[b], color=color or C["truth"], alpha=0.3,
                   lw=0, zorder=0, label=label if n == 0 else None)


def truth_mask(frame, ev):
    """Per-sample boolean ground truth aligned to `frame`, for any subset of event windows."""
    m = pd.Series(False, index=frame.index)
    for e in ev.itertuples():
        m |= ((frame["port_id"] == e.port_id) & frame["timestamp"].between(e.start, e.end))
    return m


tel["is_incident"] = truth_mask(tel, incidents)          # counts toward recall
tel["is_event"] = truth_mask(tel, events)                # any labelled window, planned included


def event_window(frame, e):
    """Boolean mask for one specific event window, whatever its planned/incident status.

    Single-event plots shade this rather than is_incident, because two of the showcase
    events (A and C) are planned changes and are deliberately absent from `incidents`.
    """
    return (frame["port_id"] == e.port_id) & frame["timestamp"].between(e.start, e.end)
print(f"{len(tel):,} rows | {len(FEATURES)} features | cadence {CADENCE:.0f}s | "
      f"{DAYS:.0f} days | {len(incidents)} incident windows, {int(events.planned.sum())} planned")

### 消融實驗（Ablation Study）

**討論。** 這一格重建的 event window 與 change calendar，決定了後面每一個 recall 的分母。分母錯了，所有成績單一起錯，而且不會有任何地方報錯。

- **Parameter：** 沒有可調的數字。`CADENCE` 與 `WINDOW` 是量出來的，不是設定的。
- **Architecture：** ground truth 從 `event_label` 這一欄重建，不從 Lab 01 的 feature 檔帶過來。這樣分母綁在 ground truth 上，不會跟著 feature 檔一起漂。
- **Per-port：** event G 與 H 各展開成五個 window，因為它們同時打在五個 port 上。第 17 步會看到，把 window 當獨立證據會灌水分母。

## 第 2 步：四種 baseline，同一個 event 四種讀法

四種 baseline 只換一件事：**拿什麼當比較對象**。score 的殼 $(x - \text{center}) / \text{scale}$ 完全不動。

| Baseline | 跟什麼比 | 結構性盲點 |
|---|---|---|
| rolling mean | 自己最近 N 筆 | 持續的 event 會稀釋自己的參考 window |
| rolling robust | 自己最近 N 筆，抗污染 | window 污染過半之後同樣被稀釋 |
| seasonal | 自己歷史上的同一個時段 | 需要歷史深度，且假設節律穩定 |
| peer | 此刻的其他 port | 對全群體同時發生的 event 盲目 |

event C（`large_file_backup`）是最乾淨的對照：它排定在每天可預測的時間，長度是 rolling window 的三倍。它同時也是一個**計畫性變更**，所以不算在 recall 的分母裡。偵測得到與該不該通知是兩件事，第 12 步才處理第二件。

In [ ]:
def floor_of(values, fraction=0.05):
    """Lower bound for any score denominator, at a fraction of the whole series' spread."""
    x = np.asarray(values, float)
    x = x[np.isfinite(x)]
    mad = float(np.median(np.abs(x - np.median(x)))) * MAD_TO_SIGMA
    if mad <= 0:
        mad = float(np.std(x)) or 0.01 * float(np.mean(np.abs(x))) or 1e-12
    return max(fraction * mad, 1e-12)


def z_of(v, center, scale):
    """Standardise to (value - center) / scale, with the floor applied and the result capped."""
    f = floor_of(v)
    return ((v - center) / scale.fillna(f).clip(lower=f)).clip(-MAX_SCORE, MAX_SCORE)


def rolling_mean(v, w):
    """Compare against the recent past, summarised by mean and standard deviation."""
    mp = max(3, w // 3)
    return v.rolling(w, min_periods=mp).mean(), v.rolling(w, min_periods=mp).std()


def rolling_robust(v, w):
    """Same comparison, but with order statistics that a few contaminated samples cannot move."""
    mp = max(3, w // 3)
    center = v.rolling(w, min_periods=mp).median()
    return center, (v - center).abs().rolling(w, min_periods=mp).median() * MAD_TO_SIGMA


def seasonal(frame, col, by_weekend=True):
    """Compare against the same slot in this series' own history, not against the last hour.

    The bucket key is weekday/weekend crossed with hour of day, so a 3am reading is
    compared with other 3am readings instead of with an evening peak that is still fading.
    This is structurally immune to recent contamination: the reference never contains
    the event being tested, however long that event runs.
    """
    v = frame[col].astype(float)
    ts = frame["timestamp"]
    daytype = np.where(ts.dt.dayofweek >= 5, "weekend", "weekday") if by_weekend else "all"
    key = pd.Series(daytype, index=frame.index) + "-" + ts.dt.hour.astype(str).str.zfill(2)
    grouped = v.groupby(key, observed=True)
    center = key.map(grouped.median())
    mad = key.map(grouped.apply(lambda s: (s - s.median()).abs().median()) * MAD_TO_SIGMA)
    return center, mad


def peer(frame, col):
    """Compare each port against its peers at the same instant, leaving itself out.

    Needs no history at all, which is why it works on a brand new port. Its blind spot is
    structural: when every peer moves together the cross-sectional median moves with them.
    """
    wide = frame.pivot_table(index="timestamp", columns="port_id", values=col, aggfunc="mean")
    m = wide.to_numpy(float)
    centers, scales = np.empty_like(m), np.empty_like(m)
    for j in range(m.shape[1]):
        others = np.delete(m, j, axis=1)             # leave-one-out: exclude the port itself
        med = np.nanmedian(others, axis=1)
        centers[:, j] = med
        scales[:, j] = np.nanmedian(np.abs(others - med[:, None]), axis=1) * MAD_TO_SIGMA
    idx = pd.MultiIndex.from_arrays([frame["timestamp"], frame["port_id"]])
    unstack = lambda arr: pd.Series(
        pd.DataFrame(arr, index=wide.index, columns=wide.columns).stack().reindex(idx).to_numpy(),
        index=frame.index)
    return unstack(centers), unstack(scales)


COL = "traffic_bps"
parts = []
for pid, g in tel.groupby("port_id", observed=True):
    g = g.sort_values("timestamp").copy()
    v = g[COL].astype(float)
    for name, (c, s) in {"rolling": rolling_mean(v, WINDOW),
                         "robust": rolling_robust(v, WINDOW),
                         "seasonal": seasonal(g, COL)}.items():
        f = floor_of(v)
        g[f"{name}_center"], g[f"{name}_scale"] = c, s.fillna(f).clip(lower=f)
        g[f"z_{name}"] = z_of(v, c, s)
    parts.append(g)
tel = pd.concat(parts).sort_values(["port_id", "timestamp"]).reset_index(drop=True)

# peer needs every port at the same timestamp, so it runs once on the combined frame.
pc, ps = peer(tel, COL)
f = floor_of(tel[COL])
tel["peer_center"], tel["peer_scale"] = pc, ps.fillna(f).clip(lower=f)
tel["z_peer"] = z_of(tel[COL], pc, ps)

BASELINES = ["rolling", "robust", "seasonal", "peer"]
print("baseline score columns:", [f"z_{b}" for b in BASELINES])

In [ ]:
# One event, four references. Left column: what each baseline calls normal, drawn as a band.
# Right column: the score that band produces, on a shared axis so the four are comparable.
# Reading the band alone is misleading, because a band that looks wide on a MB/s axis can
# still be narrow relative to the local scale, and it is the score that gets thresholded.
ev = events[events.event_id == "C"].iloc[0]
pad = pd.Timedelta("12h")
w = tel[(tel["port_id"] == ev.port_id)
        & tel["timestamp"].between(ev.start - pad, ev.end + pad)].reset_index(drop=True)
inside = event_window(w, ev)

fig, axes = plt.subplots(len(BASELINES), 2, figsize=(14, 2.1 * len(BASELINES)), sharex=True)
for (left, right), b in zip(axes, BASELINES):
    left.fill_between(w["timestamp"], (w[f"{b}_center"] - 3 * w[f"{b}_scale"]) / 1e6,
                      (w[f"{b}_center"] + 3 * w[f"{b}_scale"]) / 1e6,
                      color=C["baseline"], alpha=0.18, lw=0, label="expected range")
    left.plot(w["timestamp"], w[COL] / 1e6, color=C["signal"], lw=1.2, label="traffic")
    left.plot(w["timestamp"], w[f"{b}_center"] / 1e6, color=C["baseline"], lw=1.3, label="center")
    left.set(ylabel="MB/s", title=f"{b}: what it calls normal")
    left.legend(loc="upper left", ncol=3)

    z = w[f"z_{b}"]
    right.plot(w["timestamp"], z, color=C["score"], lw=1.2)
    right.fill_between(w["timestamp"], -THRESHOLD, THRESHOLD, color=C["muted"], alpha=0.18, lw=0)
    right.axhline(THRESHOLD, color=C["alert"], ls="--", lw=1.0)
    right.axhline(-THRESHOLD, color=C["alert"], ls="--", lw=1.0)
    right.set(ylabel="z", ylim=(-12, 12),
              title=f"{b}: {int((z.abs() > THRESHOLD)[inside].sum())} of {int(inside.sum())} "
                    f"event samples over |z| > {THRESHOLD:g}")
    for ax in (left, right):
        shade(ax, w["timestamp"], inside)
for ax in axes[-1]:
    ax.set_xlabel("time")
fig.suptitle(f"event C large_file_backup on {ev.port_id} ({int(ev.n)} samples, "
             f"{int(ev.n) / WINDOW:.0f}x the rolling window), four baselines on the same data",
             y=1.002)
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** event 持續期間，哪幾個 baseline 的 center 被拖著走？被拖走的那些，分子縮小、分母同時撐大，兩個方向都在把 score 往下壓。

- **Parameter：** `WINDOW`、floor 的 `fraction = 0.05`、`MAX_SCORE = 50`，以及畫在圖上的 ±3 倍 scale。四個都寫死。
- **Architecture：** `seasonal()` 的 bucket key 是 weekday/weekend 乘以小時。換成 day-of-week 乘以小時，bucket 變七倍細，每一格的樣本數掉到七分之一，估計更貼身也更不穩。`peer()` 的 leave-one-out 拿掉，port 自己會進到自己的參考裡，偏差往零壓。
- **Per-port：** 五個 port 共用同一個 `WINDOW`。peer 這一種還多一個前提，五個 port 必須真的可比，role 差太多時 cross-sectional 的離散度會大到什麼都抓不到。
- **Adaptive：** 把 `by_weekend` 改成 `False`，等於假設週末與平日的節律一樣。哪一個 port 最先出問題？

## 第 3 步：四種 baseline 的命中與代價

一張圖看兩件事：抓到幾個 event window，以及在正常樣本上越線的比例。沒有哪一種是全面最好，每一種都在某個可證明的案例上失敗。

In [ ]:
# Recall is counted at window level, cost at sample level outside any event. These are
# the two axes every later decision trades between.
rows = []
for b in BASELINES:
    fired = tel[f"z_{b}"].abs() > THRESHOLD
    caught = sum(fired[(tel["port_id"] == e.port_id)
                       & tel["timestamp"].between(e.start, e.end)].any()
                 for e in incidents.itertuples())
    rows.append({"baseline": b, "caught": caught,
                 "normal_over_threshold_pct": 100 * fired[~tel["is_incident"]].mean()})
compare = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 2.8), sharey=True)
y = np.arange(len(compare))
axes[0].barh(y, compare["caught"], color=C["signal"], height=0.6)
axes[0].axvline(len(incidents), color=C["truth"], lw=1.2)
for n, v in enumerate(compare["caught"]):
    axes[0].text(v + 0.2, n, str(v), va="center", fontsize=8)
axes[0].set_yticks(y, compare["baseline"])
axes[0].set(xlim=(0, len(incidents) + 2), ylabel="baseline",
            xlabel=f"incident windows caught (of {len(incidents)})", title="A. hits")

axes[1].barh(y, compare["normal_over_threshold_pct"], color=C["alert"], height=0.6)
for n, v in enumerate(compare["normal_over_threshold_pct"]):
    axes[1].text(v + 0.02, n, f"{v:.2f}%", va="center", fontsize=8)
axes[1].set(xlabel="normal samples over threshold (%)", title="B. cost")
fig.suptitle(f"one feature ({COL}), four baselines, |z| > {THRESHOLD:g}", y=1.06)
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** A 圖沒有任何一根長條碰到右邊那條線。這不是調參不夠努力，是每一種比較對象各有一類 event 結構性地看不見。

- **Parameter：** `THRESHOLD = 4.0` 同時決定了 A 圖與 B 圖。把它改成 3，命中多幾個，代價漲幾倍？
- **Architecture：** 實務的答案是同時跑多種再取最大值。取最大值以外還有投票（至少兩種同意）與加權，前者更安靜，後者需要一組沒有根據的權重。
- **Per-port：** 命中數是把五個 port 加總的。event G 與 H 各佔五個 window，所以任何在全群體事件上失效的 baseline，在這張圖上一次就掉五分。

## 第 4 步：robust 統計的 breakdown point

median 與 MAD 的關鍵性質叫 **breakdown point**：污染樣本不超過一半之前，估計值幾乎不動。mean 在第一個 outlier 就開始移動。

直接量給你看：拿一段乾淨資料，逐步把其中一定比例的樣本換成大值，看兩組估計量各自怎麼反應。

In [ ]:
# Take one quiet stretch, replace an increasing share of it with a large value, and watch
# the two estimator pairs respond. This is the breakdown point, measured rather than quoted.
clean = tel[(tel["port_id"] == ports[0]) & ~tel["is_incident"]][COL].to_numpy()[:2000]
spike = np.median(clean) * 20
fractions = np.linspace(0, 0.9, 46)

rows = []
for frac in fractions:
    x = clean.copy()
    k = int(len(x) * frac)
    x[:k] = spike                                    # contaminate the first k samples
    rows.append({"contamination": frac,
                 "mean": x.mean() / np.median(clean),
                 "median": np.median(x) / np.median(clean),
                 "std": x.std() / np.std(clean),
                 "mad": np.median(np.abs(x - np.median(x))) * MAD_TO_SIGMA / np.std(clean)})
bd = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharex=True)
axes[0].plot(bd["contamination"], bd["mean"], color=C["signal"], lw=1.6, label="mean")
axes[0].plot(bd["contamination"], bd["median"], color=C["baseline"], lw=1.6, label="median")
axes[0].set(xlabel="share of the window contaminated", ylabel="center / clean median",
            title="A. the center estimate")
axes[1].plot(bd["contamination"], bd["std"], color=C["signal"], lw=1.6, label="std")
axes[1].plot(bd["contamination"], bd["mad"], color=C["baseline"], lw=1.6, label="MAD x 1.4826")
axes[1].set(xlabel="share of the window contaminated", ylabel="scale / clean std",
            title="B. the scale estimate")
for ax in axes:
    ax.axvline(0.5, color=C["alert"], ls="--", lw=1.2, label="50% breakdown point")
    ax.legend()
fig.suptitle("robust statistics do not resist contamination gradually, they resist it "
             "completely and then fail", y=1.05)
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** robust 不等於免費。median 在乾淨資料上的 statistical efficiency 比 mean 低，也就是同樣的樣本數估出來的變異更大。任何宣稱「更 robust」的設計，都要能說出它在哪個維度付了代價。

- **Parameter：** `spike` 是乾淨資料 median 的 20 倍。改成 2 倍，breakdown point 的位置會變嗎？幅度影響的是崩潰之後跳多高，不是崩潰在哪裡發生。
- **Architecture：** median 與 MAD 的 breakdown point 是 50%，trimmed mean 由裁掉的比例決定，IQR 是 25%。要更高的 breakdown point 就要付更多 efficiency。
- **Per-port：** 這一格只拿 `ports[0]` 的一段乾淨資料。污染的形狀（連續一段 vs 散在各處）也會影響結果，這裡是連續的，也就是真實 event 的形狀。

## 第 5 步：threshold 4.0 不是單純的 tail probability

常態假設下 $|z| > 4$ 是個很小的機率，實測的越線比例遠高於它。三個效應疊加：流量分佈重尾、MAD 在安靜時段塌到下限把 score 推高、樣本之間高度 autocorrelated 使一次 event 貢獻連續好幾筆越線。

只假設變異數有限的 Chebyshev bound $P(|X - \mu| \ge k\sigma) \le 1/k^2$ 涵蓋得住實測；常態表差好幾個數量級，等於假設了資料上不存在的規律。

In [ ]:
from scipy.stats import norm

score = tel["z_robust"].abs()
t_axis = np.linspace(2, 8, 25)
observed = np.array([(score > t).mean() for t in t_axis])
gauss = 2 * (1 - norm.cdf(t_axis))                   # normal tail, both sides
cheby = 1 / t_axis ** 2                              # only assumes a finite variance

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(t_axis, observed, color=C["signal"], lw=1.8, label="measured on this data")
axes[0].plot(t_axis, cheby, color=C["baseline"], lw=1.4, ls="--", label="Chebyshev bound $1/k^2$")
axes[0].plot(t_axis, gauss, color=C["alert"], lw=1.4, ls=":", label="normal tail")
axes[0].axvline(THRESHOLD, color=C["truth"], lw=1.2)
axes[0].set_yscale("log")
axes[0].set(xlabel="threshold |z|", ylabel="share of samples over threshold",
            title="A. three answers to the same question")
axes[0].legend()

# Autocorrelation is why breaches arrive in clusters: the effective sample size is a
# fraction of the nominal one, so consecutive breaches are far commoner than independence predicts.
s1 = tel[tel["port_id"] == ports[0]][COL].reset_index(drop=True)
rho = float(s1.autocorr(lag=1))
n_eff_ratio = (1 - rho) / (1 + rho)
p = float((score > THRESHOLD).mean())
expected_pairs = p ** 2 * len(tel)
actual_pairs = int((( score > THRESHOLD) & (score.shift(1) > THRESHOLD)).sum())

axes[1].bar(["independent\nassumption", "actually\nobserved"], [expected_pairs, actual_pairs],
            color=[C["muted"], C["alert"]], width=0.55)
for n, v in enumerate([expected_pairs, actual_pairs]):
    axes[1].text(n, v, f"{v:,.0f}", ha="center", va="bottom", fontsize=9)
axes[1].set(ylabel="consecutive breach pairs",
            title=f"B. breaches cluster: {actual_pairs / expected_pairs:.1f}x more pairs than "
                  f"independence predicts\nlag-1 rho = {rho:.2f}, n_eff ~ n / {1 / n_eff_ratio:.0f}")
fig.tight_layout(); plt.show()

m = len(FEATURES) * len(ports) * len(tel) // len(ports)
print(f"{len(FEATURES)} features x {len(ports)} ports x time = {m:,} decisions this month")
print(f"Bonferroni alpha/m = {0.05 / m:.1e}  ->  |z| > {norm.ppf(1 - 0.05 / m / 2):.2f}")

### 消融實驗（Ablation Study）

**討論。** Bonferroni 給出的 threshold 比 4.0 高得多，但實務上沒有人用它。原因是它控的是「一整個月一次 false alarm 都不要有」，而值班的人真正在乎的是「每天幾則」。

- **Parameter：** A 圖的 threshold 掃到 8，B 圖的 lag 固定在 1。改用 lag 2 或 3 算 effective sample size，倍數還會更大。
- **Architecture：** Bonferroni 控的是 family-wise error rate，Benjamini-Hochberg 控的是 false discovery rate。在監控場景 FDR 比較合用，因為在意的是「發出去的 alert 裡有幾成是假的」。
- **Per-port：** rho 只量了 `ports[0]`。安靜的 port 自相關更高，effective sample size 掉得更兇。
- **Adaptive：** 把 A 圖的 score 換成 `z_seasonal` 或 `z_peer`。實測曲線離常態線更近還是更遠？把 false alarm 與漏抓合成 alerts per day 這一個量，自動吸收了 autocorrelation 與 multiple testing，而且值班的人聽得懂。

## 第 6 步：CUSUM：給偵測器記憶

上面每一種 baseline 都有同一個盲點：半個 sigma 的偏移持續不斷，逐一樣本看沒有一筆特別不尋常。z-score 的設計就是忘掉當前 window 之前的一切，它沒有「這已經偏離一陣子了」的記憶。

CUSUM 問的是**累積**的偏差夠不夠大，不是這單一讀數夠不夠遠。兩個參數各對應一個維運問題：`k` 是 deadband，多小的偏差可以完全忽略；`h` 是 decision limit，需要多少累積證據才行動。

In [ ]:
def cusum(z, k=0.5, h=5.0):
    """Two-sided cumulative sum of a deadbanded standardised residual.

    Each step adds the residual minus the deadband k and floors the running total at zero,
    so pure noise cannot creep upward, while a small persistent offset accumulates. The
    recursion carries state across samples, which is exactly what a z-score refuses to do.
    """
    z = np.nan_to_num(np.asarray(z, float))
    pos = neg = 0.0
    up, down, sig = np.zeros(len(z)), np.zeros(len(z)), np.zeros(len(z), bool)
    for i in range(1, len(z)):
        pos = max(0.0, pos + z[i] - k)               # upward arm
        neg = max(0.0, neg - z[i] - k)               # downward arm, symmetric
        up[i], down[i] = pos, neg
        if pos > h or neg > h:
            sig[i] = True
            pos = neg = 0.0                          # reset after signalling
    return pd.DataFrame({"cusum_up": up, "cusum_down": down, "cusum_signal": sig})


# Event A is a slow ramp: no single sample is extreme, which is the case CUSUM exists for.
ea = events[events.event_id == "A"].iloc[0]
pad = pd.Timedelta("3h")
w = tel[(tel["port_id"] == ea.port_id)
        & tel["timestamp"].between(ea.start - pad, ea.end + pad)].reset_index(drop=True)
w = pd.concat([w, cusum(w["z_robust"], k=0.5, h=5.0)], axis=1)

fig, axes = plt.subplots(3, 1, figsize=(13, 6.6), sharex=True)
axes[0].plot(w["timestamp"], w[COL] / 1e6, color=C["signal"], lw=1.3, label="traffic")
axes[0].plot(w["timestamp"], w["robust_center"] / 1e6, color=C["baseline"], lw=1.3,
             label="robust center")
axes[0].set(ylabel="MB/s", title="A. the baseline visibly bends toward the rising traffic")

axes[1].plot(w["timestamp"], w["z_robust"], color=C["score"], lw=1.3, label="robust z")
axes[1].axhline(THRESHOLD, color=C["alert"], ls="--", lw=1.0, label=f"threshold {THRESHOLD:g}")
axes[1].set(ylabel="z", title="B. robust z keeps triggering and clearing as the baseline catches up")

axes[2].plot(w["timestamp"], w["cusum_up"], color=C["peer"], lw=1.4, label="CUSUM upper arm")
axes[2].axhline(5.0, color=C["alert"], ls="--", lw=1.0, label="decision limit h = 5")
axes[2].set(ylabel="cumulative", xlabel="time",
            title="C. CUSUM accumulates the same residuals instead of forgetting them")

for ax in axes:
    shade(ax, w["timestamp"], event_window(w, ea))
    ax.legend(loc="upper left", ncol=3)
fig.suptitle(f"event A business_traffic_growth on {ea.port_id}", y=1.002)
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** event A 事後比對 change calendar 屬於計畫性成長，正確的處置是記錄而不是通知。偵測到與該不該通知是兩件事，第 12 步才處理第二件。

- **Parameter：** `k = 0.5` 是 deadband，`h = 5.0` 是 decision limit。把 `k` 調到 0.1，deadband 變小，純雜訊也會慢慢累積起來；把 `h` 調到 10，訊號更可信但更慢。這兩個旋鈕不能同時改善速度與 false alarm。
- **Architecture：** CUSUM 吃的 residual 現在來自 robust baseline。換成 rolling mean，baseline 被污染時 CUSUM 跟著失效。`reset_on_signal` 拿掉，一次 event 會持續觸發到訊號消失為止。
- **Per-port：** 所有 port 共用同一組 `k` 與 `h`。累積速度不同的 port，deadband 大小應該不同。
- **Adaptive：** `h` 可以從每個 port 自己的 in-control 分佈定，而不是寫死一個全域常數。

## 第 7 步：EWMA：另一種記憶

exponentially weighted moving average 對舊資料的權重按幾何級數衰減。記憶長度約 $(2-\lambda)/\lambda$ 筆，$\lambda = 0.2$ 大約是 9 筆。control limit 是 $\pm L \sigma \sqrt{\lambda / (2-\lambda)}$。

比 rolling window 多了「舊資料不是說丟就丟」的平滑，比 CUSUM 少了 deadband 這個要調的參數。代價是慢，兩次都慢：到頂比 z 晚，退回底線也比 z 晚。

In [ ]:
# Event D is a short sharp spike, the opposite of event A, so it shows EWMA's cost plainly.
ed = events[events.event_id == "D"].iloc[0]
pad = pd.Timedelta("3h")
w = tel[(tel["port_id"] == ed.port_id)
        & tel["timestamp"].between(ed.start - pad, ed.end + pad)].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(13, 3.6))
ax.plot(w["timestamp"], w["z_robust"], color=C["muted"], lw=1.2, label="robust z (no memory)")
for lam, colour in [(0.5, C["signal"]), (0.2, C["score"]), (0.05, C["peer"])]:
    e = w["z_robust"].ewm(alpha=lam, adjust=False).mean()
    limit = 3.0 * np.sqrt(lam / (2 - lam))           # L = 3 control limit for this lambda
    ax.plot(w["timestamp"], e, color=colour, lw=1.4,
            label=f"EWMA λ={lam} (memory ≈ {(2 - lam) / lam:.0f} samples, limit ±{limit:.2f})")
    ax.axhline(limit, color=colour, ls=":", lw=0.9)
ax.axhline(THRESHOLD, color=C["alert"], ls="--", lw=1.0, label=f"raw z threshold {THRESHOLD:g}")
shade(ax, w["timestamp"], event_window(w, ed))
ax.set(ylabel="score", xlabel="time",
       title="event D queue_congestion: longer memory peaks later and decays slower")
ax.legend(loc="upper left", ncol=2)
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** 三種 λ 的峰值時間與退回時間都不同。這正是記憶換來的靈敏度，代價落在同一個地方。

- **Parameter：** `lam` 決定記憶長度，`L = 3.0` 決定 control limit 的寬度。兩者要一起調，單獨動一個等於同時改了靈敏度與 false alarm 率。
- **Architecture：** CUSUM、EWMA、rolling window 三者的共同弱點是記憶裡混進了 event 本身。CUSUM 累加的 residual 是拿 rolling baseline 算的，baseline 被污染的時候三種一起失效。換偵測器解決不了輸入端的問題。
- **Per-port：** λ 對所有 port 相同。變化快的 port 應該記得短一點。

## 第 8 步：change-point detection：水位移動了嗎

到目前為止所有方法都在問「這個讀數離參考點多遠」。另一個同樣真實的問題是：底層的**水平**是不是已經永久移到另一個狀態，而不管任何單一樣本看起來極不極端。

dual moving average：短的追蹤現在，長的追蹤現在之前的近期，兩者的相對差距就是訊號。threshold 取序列自己的 p90，不從別處借。

In [ ]:
def dual_ma(v, short, long):
    """Relative gap between a fast and a slow moving average. A stable series keeps them together."""
    fast = v.rolling(short, min_periods=short // 2).mean()
    slow = v.rolling(long, min_periods=long // 2).mean()
    return (fast - slow) / (slow.abs() + 1e-9)


cp = tel[tel["port_id"] == ea.port_id].sort_values("timestamp").reset_index(drop=True)
cp["dual_ma"] = dual_ma(cp[COL], WINDOW, 6 * WINDOW)
p90 = float(cp["dual_ma"].abs().quantile(0.90))

pad = pd.Timedelta("6h")
w = cp[cp["timestamp"].between(ea.start - pad, ea.end + pad)]

fig, axes = plt.subplots(2, 1, figsize=(13, 5.0), sharex=True)
axes[0].plot(w["timestamp"], w[COL] / 1e6, color=C["signal"], lw=1.3, label="traffic")
axes[0].plot(w["timestamp"], w["robust_center"] / 1e6, color=C["baseline"], lw=1.2,
             label="robust center")
axes[0].set(ylabel="MB/s", title="A. the level steps up and the baseline follows it within an hour")
axes[1].plot(w["timestamp"], w["dual_ma"], color=C["score"], lw=1.4, label="dual MA divergence")
axes[1].plot(w["timestamp"], w["z_robust"] / 10, color=C["muted"], lw=1.0,
             label="robust z / 10, for shape comparison")
for lvl in (p90, -p90):
    axes[1].axhline(lvl, color=C["alert"], ls="--", lw=1.0)
axes[1].text(w["timestamp"].iloc[2], p90, f" its own p90 = {p90:.2f}", fontsize=8, color=C["alert"])
axes[1].set(ylabel="divergence", xlabel="time",
            title="B. the point-deviation score returns to zero while the change-point statistic peaks")
for ax in axes:
    shade(ax, w["timestamp"], event_window(w, ea))
    ax.legend(loc="upper left", ncol=3)
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** B 圖裡兩條線在同一筆資料上給出相反的讀法，因為它們問的根本是不同的問題。一個永久性的轉變與一個暫時但長期的升高，在當下看起來一模一樣。

- **Parameter：** `short = WINDOW`、`long = 6 * WINDOW`，threshold 取序列自己的 p90。把 `long` 改成 `24 * WINDOW`，延遲變長，但短暫尖峰造成的假訊號也變少。
- **Architecture：** dual moving average 是最便宜的 change-point detection。PELT 與 Bayesian online change point detection 更完整，代價是可解釋性與計算。
- **Per-port：** 這一格只示範一個 port，兩個 window 長度沒有理由所有 port 共用。
- **Adaptive：** threshold 取自序列自己的歷史，這一點已經是 adaptive 的。代價是歷史裡如果本來就有 change point，p90 會被它自己撐高。

## 第 9 步：degenerate 的欄位：統計不是工具的時候

Lab 01 第 4 步標出了大半時間為零的欄位。把這些餵進任何 z-score 家族，scale 估計會塌到下限，任何非零讀數都被一個接近零的數字除，產生巨大而無意義的 score。

fixed threshold **並非**絕對錯誤，它只在沒有自然參考點的指標上作為通用策略時是錯的。用對地方時它是本課程最好的方法，不是備案。兩個必要條件：一個有物理基礎的參考點（零錯誤是真正的零），以及正常背景與真正故障之間有數量級的差距。

In [ ]:
# Event E is the case that every statistical baseline in this notebook misses, because the
# metric carrying it is degenerate. One fixed rule catches it with no tuning at all.
ee = events[events.event_id == "E"].iloc[0]
pad = pd.Timedelta("2h")
w = tel[(tel["port_id"] == ee.port_id)
        & tel["timestamp"].between(ee.start - pad, ee.end + pad)].reset_index(drop=True)

LIMIT = 0.05
normal_max = tel.loc[~tel["is_incident"], "errors_pps"].max()

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw={"width_ratios": [1.6, 1]})
axes[0].plot(w["timestamp"], w["errors_pps"], color=C["signal"], lw=1.4, label="errors_pps")
axes[0].axhline(LIMIT, color=C["alert"], ls="--", lw=1.2, label=f"fixed rule > {LIMIT}")
shade(axes[0], w["timestamp"], event_window(w, ee))
axes[0].set(ylabel="errors/s", xlabel="time",
            title=f"A. event E link_quality_issue on {ee.port_id}")
axes[0].legend(loc="upper left")

# Separation: how far the fault sits above the highest error rate seen outside any event.
hot = tel[tel["errors_pps"] > LIMIT]
axes[1].bar(["highest outside\nany event", f"threshold", "peak during\nevent E"],
            [normal_max, LIMIT, w["errors_pps"].max()],
            color=[C["muted"], C["alert"], C["signal"]], width=0.55)
for n, v in enumerate([normal_max, LIMIT, w["errors_pps"].max()]):
    axes[1].text(n, v, f"{v:.3g}", ha="center", va="bottom", fontsize=9)
axes[1].set(ylabel="errors/s",
            title=f"B. separation: {len(hot)} samples over the rule,\n"
                  f"{100 * tel.loc[tel['errors_pps'] > LIMIT, 'is_incident'].mean():.0f}% "
                  f"of them inside an event window")
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** B 圖的三根長條之間如果沒有數量級的差距，這條規則就不成立，就會退回「挑一個數字」的老問題。條件成立與否要有人真的去驗，不能因為方便就預設使用。

- **Parameter：** `LIMIT = 0.05` 是手動設定的，跟資料裡量到的背景值有關。改成 0.01 與 0.5 各跑一次，可用的區間有多寬？區間越寬，這條規則越不需要精細調校。
- **Architecture：** 判斷一個 feature 該用 fixed threshold 還是 z-score，目前是人工判斷（看零值比例）。沒有自動切換的邏輯，新指標進來就得有人重新判斷一次。
- **Per-port：** 五個 port 共用同一個 `0.05`。不同 port 的錯誤率背景值不一定相同，值得各自校準。
- **Adaptive：** threshold 可以用每個 port 自己的歷史背景值（例如 p99）自動算出來，而不是寫死一個全域常數。

## 第 10 步：多 feature 合成 score_max

跨獨立 feature 取最大偏差，而不是平均或加總。理由是 Lab 01 第 8 步的 correlation matrix：高度相關的 feature 分開計分只是把同一份證據數兩次；取最大值讓任何一個 feature 單獨就能舉證，而不會被其他不相關的 feature 稀釋掉。

In [ ]:
# Only the columns Lab 01 marked scorable. Adding a degenerate column here would peg the
# max at the cap almost everywhere and make every later number meaningless.
SCORE_COLS = ["z_traffic_bps", "z_packets_pps", "z_avg_pkt_bytes",
              "z_broadcast_ratio", "z_multicast_ratio", "z_tx_ratio"]
tel["score_max"] = tel[SCORE_COLS].abs().max(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.4), gridspec_kw={"width_ratios": [1, 1.2]})
bins = np.linspace(0, 20, 80)
axes[0].hist(tel.loc[~tel["is_incident"], "score_max"], bins=bins, color=C["muted"],
             alpha=0.85, log=True, label=f"normal (n={int((~tel['is_incident']).sum()):,})")
axes[0].hist(tel.loc[tel["is_incident"], "score_max"], bins=bins, color=C["alert"],
             alpha=0.85, log=True, label=f"incident (n={int(tel['is_incident'].sum()):,})")
axes[0].axvline(THRESHOLD, color=C["score"], ls="--", lw=1.4, label=f"threshold {THRESHOLD:g}")
axes[0].set(xlabel="score_max (x clipped at 20)", ylabel="samples (log)",
            title="A. the two classes overlap heavily below 5")
axes[0].legend()

# Which feature actually supplied the max, inside events versus outside.
# The first samples of each port have no baseline yet, so their scores are all NaN and
# no feature "won". Fill with -1 so idxmax is defined, then drop those rows from the count.
ready = tel["score_max"].notna()
winner = tel[SCORE_COLS].abs().fillna(-1).idxmax(axis=1).str.replace("z_", "", regex=False)
inside = winner[ready & tel["is_incident"]].value_counts(normalize=True)
outside = winner[ready & ~tel["is_incident"]].value_counts(normalize=True)
order = inside.index.union(outside.index)
y = np.arange(len(order))
axes[1].barh(y + 0.19, inside.reindex(order).fillna(0), height=0.38,
             color=C["alert"], label="inside events")
axes[1].barh(y - 0.19, outside.reindex(order).fillna(0), height=0.38,
             color=C["muted"], label="outside events")
axes[1].set_yticks(y, order, fontsize=8)
axes[1].set(xlabel="share of samples where this feature supplied the max",
            title="B. which feature carries the evidence")
axes[1].legend()
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** A 圖裡兩個分佈重疊得很嚴重，沒有任何一個 threshold 能乾淨切開。這就是為什麼 score 上面還需要 confirmation 層與 policy 層，不是把 threshold 調得更聰明就好。

- **Parameter：** `SCORE_COLS` 這份名單本身就是參數。它來自 Lab 01 的 `SCORABLE`，加一欄減一欄都會動到後面每一個數字。
- **Architecture：** 把 `.max()` 換成 `.mean()`，重疊變多還是變少？平均會讓單一 feature 的強證據被其他沉默的 feature 稀釋。
- **Per-port：** B 圖顯示不同 port 主要靠哪一欄舉證。如果某個 port 幾乎只靠一欄，那一欄壞掉它就全盲。

## 第 11 步：score 到決策：連續 N 筆 confirmation

單一樣本跨過 threshold 仍然只是一個嘈雜的資料點。要求違規持續 N 個連續樣本，濾掉跨過就立刻回落的雜訊。

代價是確定的：每多要求一個樣本，就多一個取樣週期的 detection delay，$(N-1) \times$ cadence，沒有機率成分。而且 Lab 01 量到最短 event 只有 8 筆，N 不能接近那個數字。

In [ ]:
def confirm(score, threshold, n_consecutive, group):
    """Threshold a score, then require n consecutive breaches within the same port.

    Layer 1 to layer 2: a continuous score becomes a binary decision. The shift is grouped
    so a run can never straddle two ports, which would confirm an event that never happened.
    """
    breach = (score > threshold).fillna(False)
    label = breach.copy()
    for k in range(1, n_consecutive):
        label &= breach.groupby(group).shift(k).fillna(False)
    return breach, label


tel["breach"], tel["label"] = confirm(tel["score_max"], THRESHOLD, N_CONSEC, tel["port_id"])

# Sweep N to price the trade directly: how many breaches survive, and what it costs in delay.
rows = []
for n in range(1, 9):
    _, lab = confirm(tel["score_max"], THRESHOLD, n, tel["port_id"])
    caught = sum(lab[(tel["port_id"] == e.port_id)
                     & tel["timestamp"].between(e.start, e.end)].any()
                 for e in incidents.itertuples())
    rows.append({"n": n, "labels": int(lab.sum()), "caught": caught,
                 "delay_min": (n - 1) * CADENCE / 60})
conf = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2))
axes[0].plot(conf["n"], conf["labels"], marker="o", ms=5, color=C["alert"])
axes[0].set(xlabel="n consecutive", ylabel="labelled samples", title="A. noise removed")
axes[1].plot(conf["n"], conf["caught"], marker="o", ms=5, color=C["signal"])
axes[1].axhline(len(incidents), color=C["truth"], ls="--", lw=1.0, label="all incidents")
axes[1].axvline(int(events.n.min()), color=C["alert"], ls=":", lw=1.4,
                label=f"shortest event = {int(events.n.min())} samples")
axes[1].set(xlabel="n consecutive", ylabel="incident windows caught", title="B. recall")
axes[1].legend()
axes[2].plot(conf["n"], conf["delay_min"], marker="o", ms=5, color=C["score"])
axes[2].set(xlabel="n consecutive", ylabel="detection delay (min)",
            title="C. delay is deterministic, (n-1) x cadence")
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** C 圖是一條直線，因為這個延遲沒有機率成分。容易和它混淆的是另一種延遲：score 本身要花時間累積到超過 threshold。兩者在不同層次，總延遲是兩者之和，調一個不會影響另一個。

- **Parameter：** `N_CONSEC` 是唯一的旋鈕。B 圖那條垂直虛線是最短 event 的長度，把 N 設到它附近，偵測就保證會錯過最短的真實 event。這是上游測量給下游參數畫的硬性上限。
- **Architecture：** 「連續 N 筆」可以換成「最近 M 筆裡有 N 筆」，對中間掉一筆的 event 更寬容，代價是 false alarm 也更容易湊滿。Prometheus 的 `for:` 子句是同一個想法的另一種實作。
- **Per-port：** 所有 port 共用同一個 N。event 長度分佈不同的 port，上限也不同。

## 第 12 步：決策到通知：alert policy

一個 label 是統計上確認的偏差。它與「送到人手上」之間還隔著幾個過濾器，每一個都解決一個獨特的維運失靈：

- **minimum volume gate**：保護 ratio 型指標。凌晨三點 5 個封包裡有 3 個錯誤是 60% 的 error rate，不值得任何人犧牲睡眠。
- **gap tolerance**：event 中期一個邊緣樣本不該把一個 event 裂成兩則 alert。
- **change calendar suppression**：計畫性變更標記 `suppressed_by` 並且 `notified=False`，但**保留記錄**。一個默默丟棄真實 event 的規則，和一個正確抑制計畫性變更的規則，從外面看一模一樣，事後審查唯一分得出來的方法就是有一份抑制原因的紀錄。
- **severity**：偏離幅度、持續時間、資產權重的加權組合。

In [ ]:
ROLE_WEIGHT = {"wan-primary": 1.0, "server-uplink": 0.9, "wan-secondary": 0.6,
               "office-vlan": 0.5, "backup-storage": 0.4}


def runs_of(mask, gap_tolerance):
    """Contiguous True runs, merging any two runs separated by a gap no longer than the tolerance.

    This is what stops one event from fragmenting into several alerts when a single sample
    in the middle happens to fall back below the threshold.
    """
    idx = np.flatnonzero(mask)
    if idx.size == 0:
        return []
    split = np.flatnonzero(np.diff(idx) > gap_tolerance + 1)
    return list(zip(np.r_[idx[0], idx[split + 1]], np.r_[idx[split], idx[-1]]))


def severity(peak, duration_s, weight, critical_score=12.0, critical_duration_s=1800.0):
    """Blend deviation, duration and asset importance, each capped at twice its own reference.

    critical_score sits well above THRESHOLD on purpose. Set it at or below the alert
    threshold and the first term alone reaches 1.0, so every alert that clears the door is
    already 'critical' before duration or role contribute anything.
    """
    s = (0.5 * min(peak / critical_score, 2.0)
         + 0.3 * min(duration_s / critical_duration_s, 2.0)
         + 0.4 * weight)
    return ("critical" if s >= 1.0 else "warning" if s >= 0.55 else "info"), s


def build_alerts(frame, label_col="label", score_col="score_max",
                 gap_tolerance=1, min_volume_col="packets_pps", min_volume=1.0, **sev):
    """Collapse per-sample labels into alert records, carrying evidence and suppression.

    Layer 2 to layer 3. Suppressed alerts are kept in the output rather than dropped, so a
    post-incident review can tell 'never detected' apart from 'detected and deliberately silenced'.
    """
    out = []
    for (dev, pid), g in frame.groupby(["device_id", "port_id"], observed=True, sort=False):
        g = g.sort_values("timestamp")
        gate = g[label_col].fillna(False).to_numpy(bool, copy=True)
        if min_volume_col:                           # volume floor for ratio-driven alerts
            gate &= g[min_volume_col].to_numpy(float) >= min_volume
        for a, b in runs_of(gate, gap_tolerance):
            w = g.iloc[a:b + 1]
            fire, clear = w["timestamp"].iloc[0], w["timestamp"].iloc[-1]
            duration = (clear - fire).total_seconds() + CADENCE
            role = str(w["port_role"].iloc[0])
            peak = float(w[score_col].abs().max())
            sev_label, sev_score = severity(peak, duration, ROLE_WEIGHT.get(role, 0.5), **sev)
            suppressed = ""
            for ch in calendar.itertuples():
                if in_scope(ch.scope, dev, pid) and fire <= ch.end and clear >= ch.start:
                    suppressed = str(ch.change_id)
                    break
            out.append({"device_id": dev, "port_id": pid, "port_role": role,
                        "fire_time": fire, "clear_time": clear, "duration_s": duration,
                        "peak_score": peak, "severity": sev_label,
                        "severity_score": round(sev_score, 3),
                        "suppressed_by": suppressed, "notified": suppressed == ""})
    cols = ["device_id", "port_id", "port_role", "fire_time", "clear_time", "duration_s",
            "peak_score", "severity", "severity_score", "suppressed_by", "notified"]
    return (pd.DataFrame(out, columns=cols).sort_values("fire_time").reset_index(drop=True)
            if out else pd.DataFrame(columns=cols))


alerts = build_alerts(tel)
print(f"{len(alerts)} alert candidates | {int((~alerts.notified).sum())} suppressed by the "
      f"change calendar | {int(alerts.notified.sum())} notified")
display(alerts[alerts.suppressed_by != ""])

In [ ]:
# Two policy knobs swept independently, and the severity model checked by deriving its own
# behaviour from its own numbers rather than trusting the formula to be sensible.
fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.4))

vols = [1, 5, 10, 20, 50, 100]
rows = [{"min_volume": v, "alerts": len(build_alerts(tel, min_volume=v)),
         "blocked": int((tel["packets_pps"] < v).sum())} for v in vols]
mv = pd.DataFrame(rows)
axes[0].plot(mv["min_volume"], mv["alerts"], marker="o", ms=5, color=C["alert"], label="alerts")
axes[0].set(xlabel="min_volume (packets_pps)", ylabel="alerts", xscale="log",
            title=f"A. volume gate\nshipped default blocks {mv.blocked.iloc[0]} samples")

gaps = [0, 1, 2, 3, 5]
ga = pd.DataFrame([{"gap": g, "alerts": len(build_alerts(tel, gap_tolerance=g))} for g in gaps])
axes[1].plot(ga["gap"], ga["alerts"], marker="s", ms=5, color=C["signal"])
axes[1].set(xlabel="gap_tolerance (samples)", ylabel="alerts",
            title="B. gap merge: does it change anything here?")

for crit, colour in [(2.0, C["alert"]), (12.0, C["signal"])]:
    a = build_alerts(tel, critical_score=crit)
    share = a["severity"].value_counts(normalize=True).reindex(["info", "warning", "critical"]).fillna(0)
    off = -0.19 if crit == 2.0 else 0.19
    axes[2].bar(np.arange(3) + off, share.to_numpy(), width=0.38, color=colour,
                label=f"critical_score = {crit:g}")
axes[2].set_xticks(range(3), ["info", "warning", "critical"])
axes[2].set(ylabel="share of alerts", title="C. severity depends entirely on one constant")
axes[2].legend()
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** C 圖是這一步真正的結論。`critical_score = 2.0` 而 alert threshold 是 4.0，代表任何通過門檻的 alert 第一項就已經是 $0.5 \times \min(4/2, 2) = 1.0$，在持續時間與 role weight 貢獻任何東西之前就已經是 critical。一個把所有東西都標成緊急的 severity 模型，比沒有 severity 模型更糟，因為它看起來像在分類。

- **Parameter：** `gap_tolerance`、`min_volume`、`critical_score`、`critical_duration_s`，以及 severity 那三個權重 0.5 / 0.3 / 0.4。A 圖與 B 圖如果是平的，代表那個參數這個月一則 alert 都沒有影響到。一個沒在做事的參數比一個錯的參數更危險，它造成了保護已經存在的假象。把它調到 scorecard 真的動起來的第一個點。
- **Architecture：** `min_volume_col` 現在用 `packets_pps`。換成 `traffic_bps` 是另一種設計，門檻的單位與意義都不同。被 calendar 命中的 alert 是保留下來標記，不是丟掉，改成丟掉就再也分不出「沒偵測到」與「偵測到但選擇不通知」。
- **Per-port：** `ROLE_WEIGHT` 是唯一逐 role 的參數，其他都全體共用。實務上 `min_volume` 最該逐 port 設，因為它的意義本來就取決於這個 port 平常有多忙。
- **Adaptive：** severity 的三個權重加起來是 1.2 而不是 1.0，這件事沒有任何地方寫下來為什麼。在信任一個公式之前，從它的數字推導出它的實際行為。

## 第 13 步：scorecard：event recall，不是 point accuracy

base rate 0.66% 之下，一個從不觸發的偵測器 point accuracy 是 0.9934。把 accuracy 當 KPI 報告，獎勵的正是讓偵測器沉默的人。

維運真正在乎的是四個量：**event recall**（有沒有注意到）、**alerts per day**（值班負擔）、**detection delay**（多久才知道）、**duplicates per event**（理想是 1.0）。

In [ ]:
def evaluate(alerts, ev, span):
    """Operational scorecard. Recall is per event window, cost is per day, both at alert level.

    Suppressed alerts do not count as detections: silencing a planned change is a policy
    choice, not a miss, and counting it either way would hide which one happened.
    """
    live = alerts[alerts["notified"]] if len(alerts) else alerts
    rows, matched = [], set()
    for e in ev.itertuples():
        hits = live[(live["port_id"] == e.port_id) & (live["fire_time"] <= e.end)
                    & (live["clear_time"] >= e.start)]
        matched |= set(hits.index)
        first = hits["fire_time"].min() if len(hits) else pd.NaT
        rows.append({"event_id": e.event_id, "event_type": e.event_type, "port_id": e.port_id,
                     "detected": len(hits) > 0, "n_alerts": len(hits),
                     "delay_min": (first - e.start).total_seconds() / 60 if len(hits) else np.nan})
    per_event = pd.DataFrame(rows)
    days = (span[1] - span[0]).total_seconds() / 86400
    det = per_event["detected"]
    return {"event_recall": det.mean() if len(det) else 0.0,
            "detected": int(det.sum()), "total": len(per_event),
            "alerts_per_day": len(live) / days,
            "alerts_total": len(live),
            "false_alerts": len(live) - len(matched),
            "median_delay_min": per_event.loc[det, "delay_min"].median(),
            "duplicates_per_event": per_event.loc[det, "n_alerts"].mean()}, per_event


card, per_event = evaluate(alerts, incidents, SPAN)
never_fires = 1 - tel["is_incident"].mean()
point_acc = ((tel["label"] > 0) == tel["is_incident"]).mean()

fig, axes = plt.subplots(1, 2, figsize=(13.5, 3.8), gridspec_kw={"width_ratios": [1.5, 1]})
order = per_event.sort_values(["detected", "delay_min"])
colours = [C["peer"] if d else C["alert"] for d in order["detected"]]
axes[0].barh(range(len(order)), order["delay_min"].fillna(0), color=colours, height=0.62)
for n, t in enumerate(order.itertuples()):
    axes[0].text(0.3, n, "missed" if not t.detected else f"{t.delay_min:.0f} min",
                 va="center", fontsize=7, color="white" if t.detected else C["alert"])
axes[0].set_yticks(range(len(order)),
                   [f"{t.event_id} {t.event_type[:18]} {t.port_id[-4:]}" for t in order.itertuples()],
                   fontsize=7)
axes[0].set(xlabel="detection delay (minutes)",
            title=f"A. per incident: recall {card['event_recall']:.4f} "
                  f"({card['detected']}/{card['total']}), {card['alerts_per_day']:.1f} alerts/day")

axes[1].bar(["detector that\nnever fires", "this pipeline"], [never_fires, point_acc],
            color=[C["muted"], C["signal"]], width=0.55)
for n, v in enumerate([never_fires, point_acc]):
    axes[1].text(n, v, f"{v:.4f}", ha="center", va="bottom", fontsize=10)
axes[1].set(ylim=(0.98, 1.0), ylabel="point accuracy",
            title="B. why point accuracy is the wrong headline")
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** B 圖兩根長條的高低關係，就是「用 accuracy 當 KPI 會發生什麼」的完整說明。

- **Parameter：** scorecard 上四個量的定義本身就是選擇。recall 以 window 為單位，一個 event 抓到一次就算數，這對維運是對的，對「我們漏掉多少異常時刻」則不是。
- **Architecture：** alert 與 event 的配對現在用時間重疊。加一個 tolerance window，稍微遲到或提早的 alert 也算命中，recall 會上升而定義變鬆。
- **Per-port：** A 圖裡沒抓到的那幾個是誰？把它們對回 Lab 01 第 9 步的 heatmap，看是 feature 的問題還是 baseline 的問題。兩者的修法完全不同。

## 第 14 步：precision-recall 與 ROC

base rate 0.66% 之下，ROC 的 false positive rate 軸有一個巨大的 true negative 分母，幾乎把任何實際的 false alarm 數量稀釋成一個微小的比率。

$$\text{precision} = \frac{\text{TPR} \cdot \pi}{\text{TPR} \cdot \pi + \text{FPR} \cdot (1-\pi)}$$

TPR 與 FPR 都不包含 prevalence $\pi$，這就是 ROC 對不平衡不敏感的原因，也是它在這裡是負擔而不是優點的原因。

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_curve, roc_auc_score

# Drop the warm-up rows where no baseline exists yet; a curve cannot score a missing value.
ok = tel["score_max"].notna()
y = tel.loc[ok, "is_incident"].to_numpy()
s = tel.loc[ok, "score_max"].to_numpy()
base = y.mean()
prec, rec, thr = precision_recall_curve(y, s)
fpr, tpr, _ = roc_curve(y, s)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
axes[0].plot(rec, prec, color=C["signal"], lw=1.8)
axes[0].axhline(base, color=C["muted"], ls=":", lw=1.2, label=f"random = {base:.3%}")
k = int(np.searchsorted(thr, THRESHOLD))
if k < len(prec):
    axes[0].scatter([rec[k]], [prec[k]], s=45, color=C["alert"], zorder=5,
                    label=f"threshold {THRESHOLD:g}")
axes[0].set(xlabel="recall", ylabel="precision",
            title=f"A. precision-recall, AP = {average_precision_score(y, s):.3f}")
axes[0].legend()

axes[1].plot(fpr, tpr, color=C["baseline"], lw=1.8)
axes[1].plot([0, 1], [0, 1], color=C["muted"], ls=":", lw=1.0)
axes[1].set(xlabel="false positive rate", ylabel="true positive rate",
            title=f"B. ROC, AUC = {roc_auc_score(y, s):.3f}, flattering because negatives dominate")

# Same detector, same TPR and FPR, only the assumed prevalence changes.
i = int(np.argmin(np.abs(tpr - rec[max(k - 1, 0)])))
tpr_at, fpr_at = tpr[i], fpr[i]
pi = np.logspace(-3, -0.05, 100)
axes[2].plot(pi, tpr_at * pi / (tpr_at * pi + fpr_at * (1 - pi)), color=C["score"], lw=1.8)
axes[2].axvline(base, color=C["truth"], ls="--", lw=1.2, label=f"this dataset, π = {base:.3%}")
axes[2].set(xscale="log", xlabel="prevalence π", ylabel="precision",
            title="C. same TPR and FPR, precision moves with prevalence alone\n(ROC does not move at all)")
axes[2].legend()
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** C 圖裡 score 的品質一格都沒有變，動的只有假設的 prevalence。precision 從幾乎不可用走到接近完美，而 ROC 在整條線上完全不動。

*出處：Saito & Rehmsmeier (2015), [The Precision-Recall Plot Is More Informative than the ROC Plot](https://journals.plos.org/plosone/article?id=10.1371%2Fjournal.pone.0118432), PLOS ONE 10(3): e0118432。*

- **Parameter：** 沒有可調的參數，這是一張診斷圖而不是一個決定。
- **Architecture：** 兩條曲線都是 point-level 的。改成 event-level，橫軸會變成「抓到幾個 event」，縱軸變成「每則 alert 有多大機率對得上一個 event」，那才是值班的人實際感受到的量。
- **Adaptive：** 把 `s` 換成單一 feature 的 `z_traffic_bps` 再畫一次。PR 曲線掉多少，ROC 掉多少？哪一個比較誠實地反映了退步？

## 第 15 步：threshold × N 的二維 sweep

只滑動 threshold 是在一個更大的聯合設定空間裡描一條線。兩個參數一起掃，每一格都是完整的 confirm 加 build_alerts 加 evaluate。

In [ ]:
T_GRID = [2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0]
N_GRID = [1, 2, 3, 4, 5, 6]

# Every cell is a full pipeline pass, not an approximation, so the surface reflects the
# actual policy including suppression and gap merging.
rows = []
for t in T_GRID:
    for n in N_GRID:
        _, lab = confirm(tel["score_max"], t, n, tel["port_id"])
        a = build_alerts(tel.assign(label=lab))
        card_tn, _ = evaluate(a, incidents, SPAN)
        rows.append({"threshold": t, "n": n, "recall": card_tn["event_recall"],
                     "missed": card_tn["total"] - card_tn["detected"],
                     "alerts_total": card_tn["alerts_total"],
                     "per_day": card_tn["alerts_per_day"],
                     "delay": card_tn["median_delay_min"]})
grid = pd.DataFrame(rows)

from matplotlib.colors import LogNorm, Normalize


def heat(ax, col, cmap, norm, fmt, title, label):
    m = grid.pivot(index="threshold", columns="n", values=col)
    im = ax.imshow(m.to_numpy(), cmap=cmap, norm=norm, aspect="auto")
    ax.set_xticks(range(len(N_GRID)), N_GRID)
    ax.set_yticks(range(len(T_GRID)), [f"{t:g}" for t in T_GRID])
    for i, t in enumerate(T_GRID):
        for j, n in enumerate(N_GRID):
            ax.text(j, i, fmt(m.loc[t, n]), ha="center", va="center", fontsize=7.5,
                    color="#FFFFFF" if norm(m.loc[t, n]) > 0.55 else "#33383D")
    # Mark the setting every number before this step was measured at.
    ax.plot(N_GRID.index(N_CONSEC), T_GRID.index(THRESHOLD), marker="s", ms=22,
            mfc="none", mec="#1B1F23", mew=1.8)
    ax.grid(False)
    ax.set(xlabel="n consecutive", ylabel="score_max threshold", title=title)
    plt.colorbar(im, ax=ax, fraction=0.045, pad=0.03).set_label(label, fontsize=8)


fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
heat(axes[0], "recall", "YlGn", Normalize(0, 1), lambda v: f"{v:.2f}",
     "A. event recall (black square = shipped setting)", "recall")
heat(axes[1], "per_day", "OrRd", LogNorm(max(grid.per_day.min(), 1e-2), grid.per_day.max()),
     lambda v: f"{v:.1f}" if v >= 1 else f"{v:.2f}", "B. alerts per day (log colour)", "alerts/day")
heat(axes[2], "delay", "BuPu", Normalize(0, grid.delay.max()), lambda v: f"{v:.0f}",
     "C. median detection delay (min)", "minutes")
fig.tight_layout(); plt.show()

ship = grid[(grid.threshold == THRESHOLD) & (grid.n == N_CONSEC)].iloc[0]
plateau = grid[np.isclose(grid.recall, ship.recall)]
print(f"recall {ship.recall:.4f} plateau covers {len(plateau)} of {len(grid)} cells; "
      f"alerts/day ranges {plateau.per_day.min():.2f} to {plateau.per_day.max():.2f} "
      f"({plateau.per_day.max() / max(plateau.per_day.min(), 1e-9):.0f}x) at identical recall")

### 消融實驗（Ablation Study）

**討論。** 同一個 recall 高原上，alerts per day 可以差一個數量級。這種幾乎免費的改進，在只掃 threshold 的一維視角裡完全看不見。

- **Parameter：** `T_GRID` 與 `N_GRID` 的範圍決定了你看得到多大的空間。格點外面的區域不存在於這張圖上，也就不會被選中。
- **Architecture：** 兩個參數一起掃是因為它們互相影響。第三個參數（例如 `min_volume`）加進來會變成三維，格點數乘以第三軸的長度，這時就該換成隨機搜尋或貝氏最佳化。
- **Per-port：** 整個 grid 是全體 port 共用一組設定算出來的。逐 port 調參會讓 recall 更高，也讓要維護的設定變五倍。
- **Adaptive：** C 圖只跟 N 有關，跟 threshold 無關，因為 delay 是 $(N-1) \times$ cadence。這代表 A 圖與 B 圖上任何往右移的選擇，都在用 delay 換安靜，而下一步的 cost function 如果沒有為 delay 定價，就會一路把 N 推高。左上角出現的負值代表 alert 在 event 開始之前就已經開著，那是 overlap matching 的性質，不是計算錯誤。

## 第 16 步：cost function：threshold 是商業決策

$$\text{cost} = (\text{missed events}) \times R + (\text{alerts per month}), \qquad R = \frac{C_{\text{miss}}}{C_{\text{alert}}}$$

$R$ 是一個錯過的 event 值多少則 alert 的處理成本。這個數字是團隊的，不是資料的。

In [ ]:
# Sweep R over three orders of magnitude and record which grid cell minimises the cost.
R_GRID = np.unique(np.round(np.logspace(0, 3.3, 60), 1))
opt = pd.DataFrame([
    {"R": r, **grid.loc[int((grid["missed"] * r + grid["alerts_total"]).idxmin())].to_dict()}
    for r in R_GRID])

# The two-term cost never prices delay, so it can buy quiet with arbitrarily large n.
# Adding a delay term at a fixed R shows what it would take to pull n back down.
delay_rows = []
for cd in [0.0, 0.1, 0.25, 0.5, 1.0]:
    cost = (grid["missed"] * 100 + grid["alerts_total"]
            + (len(incidents) - grid["missed"]) * grid["delay"].fillna(0) * cd)
    best = grid.loc[int(cost.idxmin())]
    delay_rows.append({"C_delay": cd, "threshold": best.threshold, "n": int(best.n),
                       "recall": best.recall, "per_day": best.per_day, "delay": best.delay})
delay_opt = pd.DataFrame(delay_rows)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.0))
axes[0].step(opt["R"], opt["per_day"], where="post", color=C["signal"], lw=2.0,
             label="cost-optimal configuration")
axes[0].axhline(ship.per_day, color=C["muted"], ls="--", lw=1.4,
                label=f"shipped ({THRESHOLD:g}, {N_CONSEC})")
seen = set()
for t in opt.itertuples():
    key = (t.threshold, t.n)
    if key in seen:
        continue
    seen.add(key)
    axes[0].plot(t.R, t.per_day, "o", ms=8, color=C["signal"], mec="white", mew=1.2, zorder=3)
    axes[0].annotate(f"({t.threshold:g}, n={int(t.n)})\nrecall {t.recall:.2f}", (t.R, t.per_day),
                     textcoords="offset points", xytext=(8, -18), fontsize=7.5)
axes[0].set(xscale="log", yscale="log", xlabel="R = cost of one miss / cost of one alert",
            ylabel="alerts per day at the optimum",
            title="A. the optimum moves when the cost ratio moves")
axes[0].legend(loc="upper left")

axes[1].plot(delay_opt["C_delay"], delay_opt["n"], marker="o", ms=6, color=C["score"])
for t in delay_opt.itertuples():
    axes[1].annotate(f"thr {t.threshold:g}\n{t.per_day:.1f}/day", (t.C_delay, t.n),
                     textcoords="offset points", xytext=(6, 4), fontsize=7.5)
axes[1].set(xlabel="cost per minute of detection delay", ylabel="optimal n consecutive",
            title="B. at R = 100, pricing delay pulls n back down")
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** 出廠設定在整條 A 曲線上有落在最佳解上嗎？如果沒有，它隱含的假設是什麼？B 圖給出答案：出廠設定對應到某一個特定的 delay 代價，沒有人明確做過那個決定，但它一直都在那裡運作。

- **Parameter：** `R` 與 `C_delay`。這兩個數字是團隊的，不是資料的，而且通常從來沒有被寫下來過。
- **Architecture：** 兩項的 cost function 從來沒有為 delay 定價，所以它可以自由地用 delay 換安靜，把 N 推得任意高。加上第三項才把最佳 N 拉回來。第四項可以是「漏抓一個 wan-primary 的 event 比漏抓一個 backup-storage 的貴」。
- **Per-port：** 現在所有 port 的漏抓成本相同。不對稱成本是收束那一節的第三個問題。
- **Adaptive：** 真正的結論不是任何一個具體數字，而是沒有獨立於團隊自身 $R$ 與 delay 代價的「最佳 threshold」。任何聲稱的業界標準 threshold 都隱含假設了這兩者，而這些值可能從未被討論過。

## 第 17 步：時間 holdout 與 confidence interval

threshold 與 N 是看著這個月選的，recall 也是在同一個月量的。這個數字量化的是對這個月的擬合度，比對未見資料的表現是更弱的聲明。

在一個固定點切開月份，只在前半調參，完全在沒動過的後半評估。同時，16 個 window 背後只有 8 個獨立 event（G 與 H 各佔 5 個 port），把 window 當獨立證據會灌水分母。

In [ ]:
from scipy.stats import beta

SPLIT = tel["timestamp"].min() + (tel["timestamp"].max() - tel["timestamp"].min()) / 2
folds = {"tune H1": tel["timestamp"] < SPLIT, "test H2": tel["timestamp"] >= SPLIT,
         "full month": pd.Series(True, index=tel.index)}


def run_fold(mask, t, n):
    """Rebuild layers 2 and 3 inside one time slice and score that slice only."""
    sub = tel.loc[mask].copy()
    _, sub["label"] = confirm(sub["score_max"], t, n, sub["port_id"])
    lo, hi = sub["timestamp"].min(), sub["timestamp"].max()
    inside = incidents[incidents["start"].between(lo, hi)]
    return evaluate(build_alerts(sub), inside, (lo, hi))


fold_grid = pd.DataFrame([
    {"fold": name, "threshold": t, "n": n, **run_fold(mask, t, n)[0]}
    for name, mask in folds.items() for t in [2.5, 3.0, 4.0, 5.0, 6.0, 8.0] for n in [1, 2, 3]])


def cheapest(fold):
    """The rule a practitioner actually applies: best recall first, fewest alerts second."""
    d = fold_grid[fold_grid["fold"] == fold]
    d = d[d["event_recall"] >= d["event_recall"].max() - 1e-9]
    row = d.sort_values("alerts_per_day").iloc[0]
    return float(row["threshold"]), int(row["n"])


def cp_interval(k, n, alpha=0.05):
    """Clopper-Pearson exact binomial interval, the honest one when n is 16 or 8."""
    lo = 0.0 if k == 0 else float(beta.ppf(alpha / 2, k, n - k + 1))
    hi = 1.0 if k == n else float(beta.ppf(1 - alpha / 2, k + 1, n - k))
    return lo, hi


tuned_h1, tuned_h2 = cheapest("tune H1"), cheapest("test H2")
_, per_ev_full = run_fold(folds["full month"], THRESHOLD, N_CONSEC)
by_event = per_ev_full.groupby("event_id")["detected"].max()   # 16 windows collapse to 8 events

est = []
for label, k, n in [
        ("16 windows", int(per_ev_full["detected"].sum()), len(per_ev_full)),
        ("8 distinct events", int(by_event.sum()), len(by_event))]:
    lo, hi = cp_interval(k, n)
    est.append({"label": f"{label}  {k}/{n}", "recall": k / n, "lo": lo, "hi": hi})
est = pd.DataFrame(est)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.0))
for fold, colour in [("tune H1", C["baseline"]), ("test H2", C["signal"])]:
    d = fold_grid[fold_grid["fold"] == fold]
    axes[0].scatter(d["alerts_per_day"], d["event_recall"], s=30, alpha=0.75,
                    color=colour, label=fold)
for cfg, fold, colour in [(tuned_h1, "tune H1", C["baseline"]), (tuned_h2, "test H2", C["signal"])]:
    row = fold_grid[(fold_grid.fold == fold) & (fold_grid.threshold == cfg[0])
                    & (fold_grid.n == cfg[1])].iloc[0]
    axes[0].scatter([row["alerts_per_day"]], [row["event_recall"]], s=200, facecolors="none",
                    edgecolors=colour, lw=1.8)
    axes[0].annotate(f"cheapest best-recall on {fold}\nthreshold {cfg[0]:g}, n={cfg[1]}, "
                     f"{row['alerts_per_day']:.1f}/day",
                     (row["alerts_per_day"], row["event_recall"]), fontsize=7.5,
                     textcoords="offset points", xytext=(10, -22), color=colour)
axes[0].set(xscale="log", xlabel="alerts per day (log)", ylabel="event recall",
            title="A. the same configurations scored on each half")
axes[0].legend(loc="lower right")

ypos = np.arange(len(est))[::-1]
axes[1].errorbar(est["recall"], ypos,
                 xerr=[est["recall"] - est["lo"], est["hi"] - est["recall"]],
                 fmt="o", ms=7, lw=1.6, capsize=4, color=C["score"])
axes[1].set_yticks(ypos, est["label"], fontsize=9)
axes[1].set(xlim=(0, 1.05), xlabel="event recall, 95% Clopper-Pearson interval",
            title="B. two denominators for the same detector")
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** B 圖裡一個四位小數的 recall 意味著樣本量撐不起的精確度。在這個樣本量下，「這個東西有沒有效」的誠實版本是一個區間，不是一個小數點後四位的數字。

- **Parameter：** `SPLIT` 切在月中。換一個日期，前後兩段各自的 event 數與結論穩不穩都會變。
- **Architecture：** 這裡用的是單一時間切分。rolling-origin 或多摺交叉驗證會給出更多估計，代價是每一摺的 event 更少，區間更寬。
- **Per-port：** 切分套用到全體 port 的合併資料。個別 port 的流量模式差異大的話，也可以各自切分驗證。
- **Adaptive：** A 圖裡兩半各自最便宜的最佳解，搬到另外一半還一樣便宜嗎？holdout 揭示的不是「holdout 總是比較悲觀」，而是哪些數字在不同時期是穩定的，哪些不是。

## 第 18 步：weak label 下的評估偏差

前面每一個 recall 都用了完整的 event catalog，它在建構上是完整的。在一個真實的星期一，這個檔案不存在，你手上有的是 ticket 系統，而 ticket 只記錄了有人抱怨過的 event。

In [ ]:
# Three plausible definitions of "what counts", scored against the same alert list.
TICKETED = ["D", "F", "G"]                # users complained, certainly in the ticket system
TICKETED_PLUS = ["D", "E", "F", "G"]      # plus the one the NOC found in a link-quality report

rows = []
for name, ids in [("full catalogue", None), ("ticketed + NOC", TICKETED_PLUS),
                  ("ticketed only", TICKETED)]:
    sub = incidents if ids is None else incidents[incidents["event_id"].isin(ids)]
    got, _ = evaluate(alerts, sub, SPAN)
    rows.append({"catalogue": f"{name}\n({len(sub)} windows)", "recall": got["event_recall"],
                 "unexplained_false": got["false_alerts"]})
gt = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2), sharey=True)
y = np.arange(len(gt))
axes[0].barh(y, gt["recall"], color=C["signal"], height=0.6)
for n, v in enumerate(gt["recall"]):
    axes[0].text(v + 0.02, n, f"{v:.3f}", va="center", fontsize=9)
axes[0].set_yticks(y, gt["catalogue"], fontsize=8)
axes[0].set(xlim=(0, 1.2), xlabel="event recall", title="A. measured recall")
axes[1].barh(y, gt["unexplained_false"], color=C["alert"], height=0.6)
for n, v in enumerate(gt["unexplained_false"]):
    axes[1].text(v + 1, n, str(int(v)), va="center", fontsize=9)
axes[1].set(xlabel="unexplained false alerts", title="B. measured cost")
fig.suptitle(f"the same {card['alerts_total']} alerts, scored against three ground truths", y=1.06)
fig.tight_layout(); plt.show()

### 消融實驗（Ablation Study）

**討論。** 相同的偵測器、相同的 alert，答案純粹因為重新定義「什麼算數」而移動，而且 recall 與 false alarm 數是一起擺動的。

**系統性偏差。** 偵測器錯過的 event，往往也正是最不可能進入 ticket 系統的 event，兩者的根源相同，都是因為它隱晦。對不完整 catalog 測量的 recall 是系統性地被誇大的，而且更多的資料不會縮小這個特定的偏差。

- **Parameter：** `TICKETED` 與 `TICKETED_PLUS` 這兩份名單是假設出來的。你的環境裡對應的是工單系統實際有記錄的那些。
- **Architecture：** 三種 catalog 定義沒有哪一個是對的，它們是三種不同的問題。報告 recall 的時候要一起報告用的是哪一種定義。
- **Adaptive：** 現實中星期一早上的做法：按 peak score 排序 alert，手動裁決前 20 則。把 `alerts.sort_values("peak_score", ascending=False).head(20)` 印出來，其中有幾則對得上標記的 event window？

## 收束：threshold 是商業決策，不是工程常數

把 threshold 從 4.0 調到 3.0，多抓一個 event，代價是每月多幾百則 alert。這是一個偽裝成配置值的商業決策，通常由寫偵測程式碼的人預設做出，沒有任何對成本負責的人明確簽核。

四個值得直接向維運團隊提出的問題：

1. **交換率**：一個錯過的 event 值多少則 alert？這個數字能從人力成本推導出來，還是只是猜的？
2. **決策權**：誰設定這個權衡，記錄在哪裡，誰簽核變更？
3. **不對稱成本**：在主要 WAN 線路上錯過一個 event，與在 backup storage 上錯過一個，成本相同嗎？這條 pipeline 有沒有機制表達這個差異？
4. **退役標準**：什麼證據可以證明關掉一條規則是合理的？沒有退役條件，規則會無限期累積，而且沒有人有立場移除它們。

### 整條 pipeline 歸結成一個函式

**feature**（Lab 01）決定哪個原始量承載了故障，**baseline**（第 2 到 9 步）決定什麼是 center 與 scale，**policy**（第 11 到 12 步）決定 confirmation、suppression、severity 與通知內容。三者共同定義了一個計分函式，而它的輸出在下游看起來就是一個普通的 Prometheus metric，與硬體 counter 無法分辨。替換那個函式就是整個部署的故事，下游沒有任何東西需要改變。

每一步真正回答的問題不是「什麼是最好的演算法」，而是「什麼證據證明了這個選擇的合理性，以及你能不能在被問到時單獨為它辯護」。